
# Review Summarization Pipeline

This notebook demonstrates how to generate abstractive summaries of Amazon product reviews using transformer-based models such as PEGASUS and BART.  
The goal is to evaluate the ability of these pre-trained models to produce coherent, relevant, and concise summaries of customer sentiment.


## 1. Load and Explore the Review Dataset

We begin by importing our dataset and examining a few example reviews.
This helps us understand the structure and variety in the input data.

In [1]:
import pandas as pd

# Load Fine Food Reviews Dataset
df = pd.read_csv("../data/Reviews.csv")
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


## 2. Preprocess and Filter the Reviews

Before summarizing, we group and format the review dataset. This includes:
1. **Drop missing values**: Removes any rows that are missing `Text` (review body) or `ProductId`, as both are essential for analysis.
2. **Group by Product**: Groups the dataset by `ProductId`, then:
   - Concatenates the first 10 reviews for each product into a single string (`Text`).
   - Calculates the average review `Score` for the product.
   - Keeps the first `Summary` (title) as a placeholder reference.
3. **Rename column**: The concatenated review text is renamed from `Text` to `review_body` for clarity and consistency.
4. **Preview**: Displays a few rows of the cleaned and grouped data for inspection.

This ensures we provide the summarization models with meaningful and focused content.

In [2]:
# Drop rows missing review text or product ID
df = df.dropna(subset=['Text', 'ProductId'])

# Group by product and concatenate the first 10 reviews
df_grouped = df.groupby('ProductId').agg({
    'Text': lambda x: ' '.join(x[:10]),
    'Score': 'mean',
    'Summary': 'first'
}).reset_index()

# Rename for consistency
df_grouped.rename(columns={'Text': 'review_body'}, inplace=True)

# Preview
df_grouped[['ProductId', 'review_body', 'Score']].head()

,ProductId,review_body,Score
0,0006641040,"These days, when a person says, ""chicken soup""...",4.351351
1,141278509X,This product by Archer Farms is the best drink...,5.000000
2,2734888454,My dogs loves this chicken but its a product f...,3.500000
3,2841233731,This book is easy to read and the ingredients ...,5.000000
4,7310172001,This product is a very health snack for your p...,4.751445



## 3. Summarization with PEGASUS and BART

Here we load the PEGASUS and BART models using Hugging Face's transformers library, and apply them to summarize the sample reviews.

In [3]:
# Load PEGASUS Model and Tokenizer
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

model_name = "google/pegasus-xsum"
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

/Users/jeffreyprachick/Library/CloudStorage/OneDrive-Personal/GitHub Projects/review-summarization-pipeline/venv310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/jeffreyprachick/Library/CloudStorage/OneDrive-Personal/GitHub Projects/review-summarization-pipeline/venv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/jeffreyprachick/Library/CloudStorage/OneDrive-Personal/GitHub Projects/review-summarization-pipeline/venv310/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage wil

In [4]:
# Load BART Model and Tokenizer
from transformers import BartTokenizer, BartForConditionalGeneration

bart_model_name = "facebook/bart-large-cnn"
bart_tokenizer = BartTokenizer.from_pretrained(bart_model_name)
bart_model = BartForConditionalGeneration.from_pretrained(bart_model_name)

### Summarizing a Single Product's Reviews with PEGASUS

This block demonstrates how to generate a summary of reviews for a single product using the PEGASUS transformer model:

1. **Select Text**: Extracts the `review_body` (concatenated reviews) of the first product in the grouped dataset.
2. **Tokenization**: Uses the PEGASUS tokenizer to prepare the input text:
   - Applies truncation to fit within the model's max length (512 tokens).
   - Pads shorter inputs and returns PyTorch tensors.
3. **Generate Summary**: Feeds the tokenized input into the PEGASUS model to generate a summary:
   - Limits the summary to 60 tokens.
   - Uses beam search (`num_beams=5`) for better quality summaries.
   - Stops early when an optimal sequence is found.
4. **Decode and Display**: Converts the generated token IDs back to readable text and prints:
   - A truncated preview of the original text (first 500 characters).
   - The generated PEGASUS summary.

This serves as a proof of concept for how review summarization can be applied at the product level using transformer-based models.

In [5]:
# Summarize a Single Product's Reviews using PEGASUS
text = df_grouped['review_body'][0]

inputs = tokenizer(
    text,
    truncation=True,
    padding="longest",
    return_tensors="pt",
    max_length=512
)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=5,
    early_stopping=True
)
pegasus_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original Text:\n", text[:500], "...")
print("\nPEGASUS Summary:\n", pegasus_summary)

Original Text:
 These days, when a person says, "chicken soup" they're probably going to follow up those words with, "for the soul" or maybe "for the teenaged soul".  Didn't used to be that way.  Why I can remember a time when if a person said, "chicken soup" those words were followed by an enthusiastic "with rice!".  Such was the power of Maurice Sendak's catchy 1962 children's book.  I am pleased to report that if you care to read this book again today, you will find it hasn't dimished a jot in terms of froli ...

PEGASUS Summary:
 In our series of letters from African-American journalists, film-maker and columnist Sharon Florentine looks back at some of her favourite books.


### Summarizing the Same Review with BART

This block replicates the review summarization process using a different transformer model: **BART (Bidirectional and Auto-Regressive Transformers)**.

1. **Tokenization**: The same `review_body` text is tokenized using the BART tokenizer:
   - Truncates inputs that exceed 1024 tokens (BART's maximum).
   - Pads shorter sequences.
   - Converts the input into PyTorch tensors.
2. **Generate Summary**:
   - The BART model generates a summary using beam search (`num_beams=4`) to explore multiple generation paths.
   - The summary is capped at 60 tokens and configured to stop early once an optimal output is found.
3. **Decode and Display**:
   - The token IDs are decoded back into natural language.
   - The final BART summary is printed to allow comparison with the PEGASUS-generated version.

This highlights how different summarization models can yield varying outputs even when fed the same review content, enabling comparative evaluation.

In [6]:
# Summarize the Same Review Using BART
bart_inputs = bart_tokenizer(
    text,
    truncation=True,
    padding="longest",
    return_tensors="pt",
    max_length=1024
)

bart_summary_ids = bart_model.generate(
    bart_inputs["input_ids"],
    max_length=60,
    num_beams=4,
    early_stopping=True
)

bart_summary = bart_tokenizer.decode(bart_summary_ids[0], skip_special_tokens=True)

print("BART Summary:\n", bart_summary)

BART Summary:
 Maurice Sendak's 1962 children's book is a classic. Each month gets its own rhythmic poem and accompanying illustration. This book was the "Chicka Chicka Boom Boom" of its day, and still remains the catchiest method to teach kids the months of the year



## 4. Comparing Model Outputs

After generating the summaries, we compare the outputs side-by-side to assess content quality, coverage, and coherence.

In [7]:
# Summarize First 10 Products with PEGASUS and BART
from tqdm.auto import tqdm

results = []

# Limit to first 10 products for demo
for i, row in tqdm(df_grouped.head(10).iterrows(), total=10):
    product_id = row["ProductId"]
    review_text = row["review_body"]

    # --- PEGASUS ---
    peg_inputs = tokenizer(
        review_text,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        max_length=512
    )
    peg_ids = model.generate(
        peg_inputs["input_ids"],
        max_length=60,
        num_beams=5,
        early_stopping=True
    )
    pegasus_summary = tokenizer.decode(peg_ids[0], skip_special_tokens=True)

    # --- BART ---
    bart_inputs = bart_tokenizer(
        review_text,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        max_length=1024
    )
    bart_ids = bart_model.generate(
        bart_inputs["input_ids"],
        max_length=60,
        num_beams=4,
        early_stopping=True
    )
    bart_summary = bart_tokenizer.decode(bart_ids[0], skip_special_tokens=True)

    results.append({
        "ProductID": product_id,
        "OriginalText": review_text[:500] + "...",  # Truncated for display
        "PEGASUS_Summary": pegasus_summary,
        "BART_Summary": bart_summary
    })

# Convert to DataFrame
summary_df = pd.DataFrame(results)

# Save to CSV
summary_df.to_csv("../outputs/model_comparison_summaries.csv", index=False)

# Display
summary_df.head()

100%|███████████████████████████████████████████| 10/10 [01:22<00:00,  8.26s/it]


,ProductID,OriginalText,PEGASUS_Summary,BART_Summary
0,0006641040,"These days, when a person says, ""chicken soup""...",In our series of letters from African-American...,Maurice Sendak's 1962 children's book is a cla...
1,141278509X,This product by Archer Farms is the best drink...,Have you ever wondered what it would be like t...,This product by Archer Farms is the best drink...
2,2734888454,My dogs loves this chicken but its a product f...,Is it safe to buy chicken products made in the...,My dogs loves this chicken but its a product f...
3,2841233731,This book is easy to read and the ingredients ...,I have been using this book to make healthy sn...,This book is easy to read and the ingredients ...
4,7310172001,This product is a very health snack for your p...,Our Premium Beef Liver Training Treats are a d...,This product is a very health snack for your p...


## Key Observations

The following table summarizes the qualitative differences observed between the two transformer-based summarization models:

| **Model**          | **Summary Behavior**                                                                                                                                       |
|--------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **PEGASUS (XSUM)** | Hallucinated content unrelated to the review (e.g., mentions of “African-American journalists” and “Sharon Florentine”). Common issue when inputs don’t resemble news articles. |
| **BART (CNN)**     | Much more **grounded**, **faithful**, and **interpretable**. It accurately captures the book, the monthly rhythm, and the context of the original review. |

These results reflect how domain mismatch affects the quality of summarization. PEGASUS (trained on extreme summarization tasks) may generate fluent but factually incorrect summaries when applied to non-news content. In contrast, BART tends to produce more reliable and context-aware outputs.